# 🧠 Implementasi Arsitektur 1: Recurrent Neural Network (Bidirectional GRU)
### Klasifikasi Multi-Kelas Teks Aduan Publik Smart City Pemkot Makassar
> **Tugas Individu Mata Kuliah Deep Learning**  
> **Nama Mahasiswa:** Haris  
> **NIM:** H071241070  
> **Program Studi:** Sistem Informasi, FMIPA Universitas Hasanuddin  
> **Dosen Pengampu:** Tim Dosen Deep Learning  

---

### 📌 Ringkasan Eksperimen Arsitektur 1 (RNN):
1. **Konsep Arsitektur:** Memanfaatkan Recurrent Neural Network dengan sel **Gated Recurrent Unit (GRU)** dua arah (*Bidirectional GRU*) yang dirancang untuk menangani ketergantungan sekuensial teks (*temporal dependency*) serta mengatasi masalah *vanishing gradient*.
2. **Komponen Model:**
   * **Embedding Layer:** Memetakan indeks kata ke representasi vektor berdimensi 128 (`embedding_dim = 128`).
   * **Spatial/Feature Dropout (0.3):** Mencegah ko-adaptasi berlebih antar fitur embedding.
   * **Bidirectional GRU (2 Layers):** `hidden_dim = 128` per arah (total 256 output forward-backward).
   * **Global Max Pooling 1D:** Mengekstrak representasi fitur paling dominan sepanjang sekuens kalimat.
   * **Dense Classification Head:** Fully connected layers dengan aktivasi ReLU, Dropout (0.4), dan Linear layer 5 kelas.
3. **Penanganan Class Imbalance:** Menggunakan *Class-Weighted Cross Entropy Loss* agar penalti kelas minoritas (Jalan Rusak & Sampah) berbobot proporsional.
4. **Metrik Evaluasi Utama:** **Macro F1-Score** dan Akurasi Global pada 1.508 data uji (*unseen test set*).

## 1. Setup Lingkungan & Deteksi Akselerasi Hardware (GPU / CPU)
Mendeteksi apakah runtime berjalan pada Google Colab dengan GPU atau CPU lokal, serta mengatur seed acak untuk reproducibility.

In [ ]:
import os
import sys
import time
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

# Set Random Seed untuk Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device yang aktif: {device}")
if torch.cuda.is_available():
    print(f"🔥 GPU Model     : {torch.cuda.get_device_name(0)}")
    print(f"💾 Total VRAM GPU : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("ℹ️ Berjalan pada CPU.")

## 2. Pemuatan Dataset Bersih Hasil Stratified Split (80:10:10)
Memuat file `data_clean_train.csv`, `data_clean_val.csv`, dan `data_clean_test.csv` yang tersimpan di direktori `../data/`.

In [ ]:
# Lokasi direktori dataset
data_paths = [
    '../data',
    './data',
    '/content/data',
    'D:/Kulyeah/Semester 5/Deep Learning/Tugas_DeepLearning_H071241070/data',
    'D:/Kulyeah/Semester 5/RSIC'
]

DATA_DIR = None
for p in data_paths:
    if os.path.exists(os.path.join(p, 'data_clean_train.csv')):
        DATA_DIR = p
        break

if DATA_DIR is None:
    raise FileNotFoundError("Dataset CSV tidak ditemukan. Pastikan data_clean_train.csv berada di folder ../data/")

train_df = pd.read_csv(os.path.join(DATA_DIR, 'data_clean_train.csv'))
val_df = pd.read_csv(os.path.join(DATA_DIR, 'data_clean_val.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'data_clean_test.csv'))

print(f"✅ Data Train : {len(train_df):,} baris")
print(f"✅ Data Val   : {len(val_df):,} baris")
print(f"✅ Data Test  : {len(test_df):,} baris")

# Mapping label kategori ke integer ID
categories = sorted(train_df['Kategori'].unique().tolist())
label2id = {cat: idx for idx, cat in enumerate(categories)}
id2label = {idx: cat for cat, idx in label2id.items()}

print("\nPemetaan Kelas Kategori:")
for cat, idx in label2id.items():
    print(f"  ID {idx}: {cat}")

## 3. Pembangunan Kosakata (Vocabulary) & Tokenisasi Teks
Membangun kamus indeks kata (*Word2Index*) dari data latih, memetakan token khusus `<PAD>` dan `<UNK>`, serta menetapkan batas panjang sekuens maksimum 128 kata.

In [ ]:
from collections import Counter

# Tokenisasi kata sederhana berbasis spasi
def tokenize(text):
    return str(text).lower().split()

# Hitung frekuensi kata pada data train
word_counts = Counter()
for text in train_df['clean_text']:
    word_counts.update(tokenize(text))

# Bangun kosakata dengan frekuensi kemunculan minimal 2 kali
PAD_TOKEN = '<PAD>'
UNK_TOKEN = '<UNK>'

vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
min_freq = 2
for word, count in word_counts.items():
    if count >= min_freq:
        vocab[word] = len(vocab)

print(f"Total kata unik dalam korpus : {len(word_counts):,}")
print(f"Ukuran kosakata (Vocab Size) : {len(vocab):,} kata (min_freq={min_freq})")

MAX_LEN = 128

def text_to_indices(text, vocab, max_len=MAX_LEN):
    tokens = tokenize(text)
    indices = [vocab.get(w, vocab[UNK_TOKEN]) for w in tokens][:max_len]
    # Padding jika kurang dari max_len
    if len(indices) < max_len:
        indices += [vocab[PAD_TOKEN]] * (max_len - len(indices))
    return indices

# Contoh representasi token
sample_text = train_df['clean_text'].iloc[0]
sample_indices = text_to_indices(sample_text, vocab)
print(f"\nTeks Asli : {sample_text[:80]}...")
print(f"Indeks    : {sample_indices[:15]}...")

## 4. Konstruksi PyTorch Dataset & DataLoader

In [ ]:
class AduanDataset(Dataset):
    def __init__(self, df, vocab, label2id, max_len=128):
        self.texts = df['clean_text'].values
        self.labels = [label2id[k] for k in df['Kategori'].values]
        self.vocab = vocab
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
        
    def __getitem__(self, idx):
        indices = text_to_indices(self.texts[idx], self.vocab, self.max_len)
        label = self.labels[idx]
        return {
            'input_ids': torch.tensor(indices, dtype=torch.long),
            'label': torch.tensor(label, dtype=torch.long)
        }

BATCH_SIZE = 32

train_dataset = AduanDataset(train_df, vocab, label2id, max_len=MAX_LEN)
val_dataset = AduanDataset(val_df, vocab, label2id, max_len=MAX_LEN)
test_dataset = AduanDataset(test_df, vocab, label2id, max_len=MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Batch per Epoch (Train): {len(train_loader)}")

## 5. Perancangan Arsitektur Model Bidirectional GRU (BiGRU)
Arsitektur model mengintegrasikan:
1. `nn.Embedding(vocab_size, embedding_dim=128, padding_idx=0)`
2. `Spatial Dropout` 0.3
3. `nn.GRU(128, 128, num_layers=2, bidirectional=True, batch_first=True, dropout=0.3)`
4. `Global Max Pooling 1D` untuk menangkap fitur paling menonjol
5. Fully Connected Classifier Head dengan ReLU dan Dropout 0.4

In [ ]:
class BiGRUTextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, num_layers=2, num_classes=5, dropout=0.3):
        super(BiGRUTextClassifier, self).__init__()
        
        # 1. Embedding Layer
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.embed_dropout = nn.Dropout(dropout)
        
        # 2. Bidirectional GRU Layer
        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        # 3. Dense Classifier Head
        self.fc1 = nn.Linear(hidden_dim * 2, 128)
        self.relu = nn.ReLU()
        self.fc_dropout = nn.Dropout(0.4)
        self.fc2 = nn.Linear(128, num_classes)
        
    def forward(self, input_ids):
        # Shape: (batch_size, seq_len) -> (batch_size, seq_len, embed_dim)
        embedded = self.embed_dropout(self.embedding(input_ids))
        
        # Shape: (batch_size, seq_len, hidden_dim * 2)
        gru_out, _ = self.gru(embedded)
        
        # Global Max Pooling sepanjang dimensi waktu sekuens (dim=1)
        # Shape: (batch_size, hidden_dim * 2)
        pooled, _ = torch.max(gru_out, dim=1)
        
        # Fully connected head
        x = self.fc_dropout(self.relu(self.fc1(pooled)))
        logits = self.fc2(x)
        return logits

model_bigru = BiGRUTextClassifier(
    vocab_size=len(vocab),
    embed_dim=128,
    hidden_dim=128,
    num_layers=2,
    num_classes=len(categories),
    dropout=0.3
).to(device)

total_params = sum(p.numel() for p in model_bigru.parameters() if p.requires_grad)
print(model_bigru)
print(f"\nTotal Parameter Terlatih (Trainable Parameters): {total_params:,}")

## 6. Penanganan Ketimpangan Kelas (Class Weights) & Optimizer Adam

In [ ]:
# Hitung bobot kelas berbanding terbalik dengan frekuensi kemunculan
class_counts = train_df['Kategori'].value_counts()
weights = []
for idx in range(len(categories)):
    cat_name = id2label[idx]
    cnt = class_counts.get(cat_name, 1)
    weights.append(len(train_df) / (len(categories) * cnt))

class_weights_tensor = torch.tensor(weights, dtype=torch.float).to(device)
print("Bobot Penalti Kelas (Class Weights):")
for idx, w in enumerate(weights):
    print(f"  {id2label[idx]:<22}: {w:.4f}")

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.Adam(model_bigru.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=1, verbose=True)

## 7. Proses Pelatihan Model (Training Loop) & Evaluasi Validasi

In [ ]:
EPOCHS = 5
best_val_f1 = 0.0
history = {'train_loss': [], 'val_loss': [], 'val_macro_f1': []}

print(f"🚀 Memulai Pelatihan Model BiGRU selama {EPOCHS} Epochs...")
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    # --- TRAINING PHASE ---
    model_bigru.train()
    total_train_loss = 0.0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        labels = batch['label'].to(device)
        
        optimizer.zero_grad()
        logits = model_bigru(input_ids)
        loss = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model_bigru.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_train_loss += loss.item()
        
    avg_train_loss = total_train_loss / len(train_loader)
    
    # --- VALIDATION PHASE ---
    model_bigru.eval()
    total_val_loss = 0.0
    val_preds, val_targets = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            labels = batch['label'].to(device)
            logits = model_bigru(input_ids)
            loss = criterion(logits, labels)
            total_val_loss += loss.item()
            
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_targets.extend(labels.cpu().numpy())
            
    avg_val_loss = total_val_loss / len(val_loader)
    val_macro_f1 = f1_score(val_targets, val_preds, average='macro') * 100
    
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['val_macro_f1'].append(val_macro_f1)
    
    print(f"Epoch [{epoch}/{EPOCHS}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Macro F1: {val_macro_f1:.2f}%")
    
    scheduler.step(val_macro_f1)
    if val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        torch.save(model_bigru.state_dict(), 'best_bigru_model.pt')
        print(f"  ⭐ Checkpoint tersimpan! (Val Macro F1 Terbaik: {best_val_f1:.2f}%)")

print(f"\n🏁 Pelatihan Selesai dalam {(time.time() - start_time):.2f} detik!")

## 8. Visualisasi Kurva Loss & Macro F1

In [ ]:
plt.figure(figsize=(12, 4), dpi=120)

plt.subplot(1, 2, 1)
plt.plot(range(1, EPOCHS + 1), history['train_loss'], marker='o', label='Training Loss', color='#2563EB')
plt.plot(range(1, EPOCHS + 1), history['val_loss'], marker='s', label='Validation Loss', color='#DC2626')
plt.title('Kurva Loss Pelatihan vs Validasi (BiGRU)', fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(range(1, EPOCHS + 1), history['val_macro_f1'], marker='^', label='Validation Macro F1', color='#059669')
plt.title('Kurva Evaluasi Macro F1-Score (BiGRU)', fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Macro F1 (%)')
plt.legend()

plt.tight_layout()
plt.show()

## 9. Evaluasi Kinerja pada Data Pengujian (Test Set - 1.508 Sampel)

In [ ]:
# Muat bobot model terbaik jika tersedia
if os.path.exists('best_bigru_model.pt'):
    model_bigru.load_state_dict(torch.load('best_bigru_model.pt'))

model_bigru.eval()
test_preds, test_targets = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        logits = model_bigru(input_ids)
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        test_preds.extend(preds)
        test_targets.extend(batch['label'].numpy())

bigru_acc = accuracy_score(test_targets, test_preds) * 100
bigru_macro_f1 = f1_score(test_targets, test_preds, average='macro') * 100
bigru_weighted_f1 = f1_score(test_targets, test_preds, average='weighted') * 100

print("=" * 68)
print("  HASIL EVALUASI MODEL ARSITEKTUR 1: RNN (BIDIRECTIONAL GRU)")
print("=" * 68)
print(f"Akurasi Global    : {bigru_acc:.2f}%")
print(f"Macro F1-Score    : {bigru_macro_f1:.2f}% (Metrik Utama Ketimpangan Kelas)")
print(f"Weighted F1-Score : {bigru_weighted_f1:.2f}%")
print("=" * 68 + "\n")

target_names = [id2label[i] for i in range(len(categories))]
print(classification_report(test_targets, test_preds, target_names=target_names, digits=4))

# Confusion Matrix Heatmap
cm = confusion_matrix(test_targets, test_preds)
plt.figure(figsize=(7, 5), dpi=120)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.title("Confusion Matrix: RNN (BiGRU) pada Test Set", fontweight='bold', pad=10)
plt.xlabel("Prediksi Model")
plt.ylabel("Aktual (Ground Truth)")
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

## 10. Pengukuran Kecepatan Inferensi (Latency Benchmark)

In [ ]:
# Benchmark waktu komputasi 50 iterasi
sample_input = text_to_indices(test_df['clean_text'].iloc[0], vocab)
input_tensor = torch.tensor([sample_input], dtype=torch.long).to(device)

model_bigru.eval()
with torch.no_grad():
    # Warm-up
    for _ in range(5):
        _ = model_bigru(input_tensor)
        
    start_t = time.time()
    n_iters = 50
    for _ in range(n_iters):
        _ = model_bigru(input_tensor)
    latency_bigru = ((time.time() - start_t) / n_iters) * 1000

print(f"⚡ Rata-rata Latensi Inferensi BiGRU: {latency_bigru:.2f} ms / aduan")

## 11. Simulasi Prediksi Teks Aduan Baru Warga

In [ ]:
def prediksi_aduan_bigru(teks):
    model_bigru.eval()
    indices = text_to_indices(teks, vocab, max_len=MAX_LEN)
    t = torch.tensor([indices], dtype=torch.long).to(device)
    with torch.no_grad():
        logits = model_bigru(t)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        pred_idx = np.argmax(probs)
        conf = probs[pred_idx] * 100
        
    print(f"Pesan Warga: \"{teks}\"")
    print(f"Prediksi   : 📂 {id2label[pred_idx]} (Keyakinan: {conf:.2f}%)\n")

contoh_aduan = [
    "Lampu jalan di perintis kemerdekaan padam gelap sekali rawan begal",
    "Pohon besar tumbang di pettarani menimpa tiang kabel bahaya",
    "Got depan pasar daya meluap tersumbat sampah kotor dan bau busuk",
    "Jalanan berlubang di hertasning banyak pemotor jatuh tolong aspal"
]

for c in contoh_aduan:
    prediksi_aduan_bigru(c)